# Week 37, Day 1: Neurodata Without Borders: how neural data are stored

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w037/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

### Exercise: Walking the file

Complete visit(name, obj) so that, for every dataset in the file, tree maps its full path (with a leading slash) to its shape. f.visititems calls visit on every group and dataset below the root.

In [ ]:
import io
import numpy as np
import h5py
import matplotlib.pyplot as plt
r = np.random.default_rng(2022)

# --- Simulate a session: 300 trials, 24 units in 3 brain areas ---
n_trials, n_units = 300, 24
starts = np.cumsum(r.uniform(2.5, 3.5, n_trials))
stops = starts + 2.0
contrast = r.choice([-1, -0.5, -0.25, 0, 0.25, 0.5, 1], n_trials)      # negative = left
choice = np.where(r.random(n_trials) < 1 / (1 + np.exp(-6 * contrast)), 1, -1)
areas = np.repeat(["VISp", "MOs", "CA1"], 8)
spike_lists = []
for u in range(n_units):
    base = r.uniform(2, 10)
    times = list(r.uniform(0, stops[-1] + 2, r.poisson(base * (stops[-1] + 2))))
    for s, c, ch in zip(starts, contrast, choice):
        if areas[u] == "VISp":                      # visual response 50 to 250 ms after onset
            times += list(s + 0.5 + r.uniform(0.05, 0.25, r.poisson(4 * abs(c) + 0.2)))
        if areas[u] == "MOs":                       # choice signal before the response
            times += list(s + 0.8 + r.uniform(0, 0.4, r.poisson(3 if ch == 1 else 0.3)))
    spike_lists.append(np.sort(np.array(times)))

# --- Write it with the layout of an NWB file ---
buf = io.BytesIO()
with h5py.File(buf, "w") as w:
    w.attrs["session_description"] = "Simulated visual decision task"
    w.attrs["identifier"] = "engram-w037-demo"
    w.create_dataset("general/subject/species", data="Mus musculus")
    w.create_dataset("general/subject/subject_id", data="mouse-01")
    tr = w.create_group("intervals/trials")
    tr.create_dataset("start_time", data=starts); tr.create_dataset("stop_time", data=stops)
    tr.create_dataset("contrast", data=contrast); tr.create_dataset("choice", data=choice)
    tr.create_dataset("stim_on_time", data=starts + 0.5)
    el = w.create_group("general/extracellular_ephys/electrodes")
    el.create_dataset("location", data=np.array(areas, dtype="S8"))
    un = w.create_group("units")
    un.create_dataset("spike_times", data=np.concatenate(spike_lists))
    un.create_dataset("spike_times_index", data=np.cumsum([len(s) for s in spike_lists]))
    un.create_dataset("electrodes", data=np.arange(n_units))
    un.create_dataset("id", data=np.arange(n_units))
f = h5py.File(buf, "r")

tree = {}
def visit(name, obj):
    ...
f.visititems(visit)
for k, v in sorted(tree.items()):
    print(f"{k:45s} {v}")

In [ ]:
def test_tree():
    "tree lists every dataset with its full path and shape"
    assert tree["/units/spike_times"] == (len(f["/units/spike_times"]),)
    assert tree["/intervals/trials/choice"] == (n_trials,)
    assert all(isinstance(v, tuple) for v in tree.values())
    assert len(tree) >= 12

test_tree()
print("All checks passed")

### Exercise: Reading one unit

Write spikes_of(f, k), returning unit k's spike times by slicing f['/units/spike_times'] with the boundaries in f['/units/spike_times_index']. Read only the slice you need.

In [ ]:
def spikes_of(f, k):
    idx = f["/units/spike_times_index"]
    start = ...
    stop = ...
    return f["/units/spike_times"][start:stop]

loc = [s.decode() for s in f["/general/extracellular_ephys/electrodes/location"][:]]
for k in (0, 8, 16):
    s = spikes_of(f, k)
    print(f"unit {k:2d} in {loc[k]}: {len(s)} spikes, {len(s) / s[-1]:.1f} Hz")

In [ ]:
import numpy as np
def test_first():
    "unit 0 is read correctly"
    assert np.allclose(spikes_of(f, 0), spike_lists[0])

def test_all():
    "every unit is read correctly"
    assert all(np.allclose(spikes_of(f, k), spike_lists[k]) for k in range(n_units))

test_first()
test_all()
print("All checks passed")

### Exercise: PSTHs by area and choice

Write psth(spikes, events): for each event, count the spikes in the bins given by edges relative to the event, then return the mean count per trial divided by the bin width (a rate in Hz). Then compute the MOs population PSTH separately for rightward (1) and leftward (-1) choices.

In [ ]:
edges = np.arange(-0.5, 1.0001, 0.025); centres = (edges[:-1] + edges[1:]) / 2

def psth(spikes, events):
    counts = ...
    return counts.mean(0) / np.diff(edges)

on = f["/intervals/trials/stim_on_time"][:]
ch = f["/intervals/trials/choice"][:]
mos = {c: np.mean([psth(spikes_of(f, k), on[ch == c]) for k in range(8, 16)], 0) for c in (1, -1)}
vis = np.mean([psth(spikes_of(f, k), on) for k in range(8)], 0)
plt.plot(centres, vis, label="VISp, all trials")
plt.plot(centres, mos[1], label="MOs, right choices"); plt.plot(centres, mos[-1], label="MOs, left choices")
plt.axvline(0, color="gray", lw=0.5); plt.xlabel("time from stimulus onset (s)"); plt.ylabel("rate (Hz)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_shape():
    "psth returns one rate per bin"
    assert psth(spikes_of(f, 0), f["/intervals/trials/stim_on_time"][:]).shape == (len(edges) - 1,)

def test_visual():
    "VISp units respond after stimulus onset: the rate 50 to 250 ms after onset exceeds the rate before it"
    on = f["/intervals/trials/stim_on_time"][:]
    p = np.mean([psth(spikes_of(f, k), on) for k in range(8)], 0)
    assert p[(centres > 0.05) & (centres < 0.25)].mean() > 1.5 * p[centres < 0].mean()

def test_choice():
    "MOs units fire at least 1.5 times as fast before rightward choices as before leftward ones"
    assert mos[1][(centres > 0.3) & (centres < 0.7)].mean() > 1.5 * mos[-1][(centres > 0.3) & (centres < 0.7)].mean()

test_shape()
test_visual()
test_choice()
print("All checks passed")